# Enterprise World Model Engine (`ewm-engine` v1.5.0) — Interactive Quickstart Tutorial

This executable tutorial walks through the core audited capabilities of `ewm-engine` (`EWM-001` through `EWM-006`):
1. **Strict State Immutability, Hybrid Dynamics Routing & `do()` Interventions (`EWM-001`–`EWM-003`)**
2. **Finance Domain Pack (`EWM-004`)**: Exact `decimal.Decimal` double-entry accounting ($\sum \text{Assets} - \sum \text{Liabilities} - \sum \text{Equity} = \text{Decimal('0.00')}$) and multi-step liquidity drain stress testing.
3. **Software Engineering (SWE) Domain Pack (`EWM-005`)**: `nx.DiGraph` dependency DAG enforcement, $O(V+E)$ cycle rejection, downstream build queue propagation, and concurrent agent `do()` interventions.
4. **Scientific Evaluation Framework (`EWM-006`)**: `EWM_Evaluator` comparing `RandomAgent`, `StaticRuleAgent`, and `EWMPolicyAgent` across $N=30$ seeds with Stratified Interquartile Mean (IQM) and 95% bootstrap confidence intervals.

## 1. Core Kernel: Immutable `WorldState`, `TransitionEngine`, `do()` & Conformal Abstention (`EWM-001`–`EWM-003`)

In [1]:
import numpy as np
import ewm_engine as ewm
from ewm_engine.causal import CausalGraph, TransitionEngine
from ewm_engine.dynamics.calibration import (
    AbstainException,
    MahalanobisOODAndDriftDetector,
    Predictor,
)
from ewm_engine.dynamics.learned import (
    LinearResidualDynamics,
    TransitionDataset,
    TransitionSample,
)

# 1. Initialize an immutable WorldState with bounded resources
root_state = ewm.WorldState(
    resources=[
        ewm.Resource(id="stock_a", current=100.0, min_value=0.0, max_value=200.0),
        ewm.Resource(id="stock_b", current=20.0, min_value=0.0, max_value=200.0),
    ]
)
root_hash = root_state.state_hash

# 2. Route an explicit do() intervention through a branched TransitionEngine
graph = CausalGraph()
for node in ("stock_a", "stock_b"):
    graph.add_node(node)
graph.add_edge("stock_a", "stock_b")

engine = TransitionEngine(
    initial_state=root_state,
    causal_graph=graph,
    structural_equations={
        "stock_a": lambda parents, actions, u: parents.get("stock_a", 100.0),
        "stock_b": lambda parents, actions, u: 120.0 - parents.get("stock_a", 100.0),
    },
)
outcome = (
    engine.branch("counterfactual_do_a")
    .do(ewm.Action(id="act_do_a", type="do", parameters={"target": "stock_a", "value": 70.0}))
    .step()
)
assert root_state.state_hash == root_hash  # Zero cross-branch mutation
assert outcome.next_state.get_resource("stock_a").current == 70.0
assert outcome.next_state.get_resource("stock_b").current == 50.0
assert outcome.is_deterministic is True

# 3. Wrap a learned predictor in Conformal calibration + Mahalanobis OODDetector
rng = np.random.default_rng(42)
samples = []
for i in range(60):
    s_a = float(90.0 + rng.normal(0.0, 2.0))
    s_b = float(30.0 + rng.normal(0.0, 2.0))
    s_t = ewm.WorldState(
        step=i,
        resources=[
            ewm.Resource(id="stock_a", current=s_a, min_value=0.0, max_value=200.0),
            ewm.Resource(id="stock_b", current=s_b, min_value=0.0, max_value=200.0),
        ],
    )
    act = ewm.Action(id=f"act_{i}", type="rebalance", parameters={"value": 1.0})
    s_next = ewm.WorldState(
        step=i + 1,
        resources=[
            ewm.Resource(id="stock_a", current=s_a - 2.0, min_value=0.0, max_value=200.0),
            ewm.Resource(id="stock_b", current=s_b + 2.0, min_value=0.0, max_value=200.0),
        ],
    )
    samples.append(TransitionSample(state=s_t, actions=(act,), events=(), next_state=s_next))

train_ds = TransitionDataset(samples[:40])
cal_ds = TransitionDataset(samples[40:])
model = LinearResidualDynamics(target_resource="stock_a", action_type="rebalance")
model.fit(train_ds)

ood_detector = MahalanobisOODAndDriftDetector(regularization=1e-5)
ood_detector.fit(train_ds, tune_quantile=0.99, safety_margin=1.20)
predictor = Predictor(
    models=[model],
    ood_detector=ood_detector,
    ood_threshold=ood_detector.ood_threshold,
    threshold=25.0,
    nominal_coverage=0.90,
)
predictor.calibrate(cal_ds, alpha=0.10)

pred, ci = predictor.predict(cal_ds[0].state, cal_ds[0].actions)
assert ci.max_width <= 25.0

# Out-of-distribution state triggers AbstainException
ood_state = ewm.WorldState(
    resources=[
        ewm.Resource(id="stock_a", current=199.0, min_value=0.0, max_value=200.0),
        ewm.Resource(id="stock_b", current=199.0, min_value=0.0, max_value=200.0),
    ]
)
abstained = False
try:
    predictor.predict(ood_state)
except AbstainException:
    abstained = True
assert abstained is True
print("Section 1 verified: Immutable branching, TransitionEngine do(), and OOD abstention passed.")

## 2. Finance Domain Pack (`EWM-004`): Exact `Decimal` Double-Entry Ledger & Liquidity Stress Test

In [2]:
from decimal import Decimal
from ewm_engine.domains.finance import (
    ZERO_DECIMAL,
    FinancialState,
    build_corporate_treasury_engine,
    build_journal_action,
    run_liquidity_drain_stress_test,
)

# 1. Initialize corporate treasury engine ($40.0M balanced assets = liabilities + equity)
fin_engine, fin_state = build_corporate_treasury_engine()
assert fin_state.is_balanced is True
assert fin_state.accounting_discrepancy == ZERO_DECIMAL

# 2. Execute a balanced $250,000.75 loan repayment
outcome_repay = fin_engine.step(
    action=build_journal_action(
        "valid_loan_repayment",
        debits={"cash": Decimal("250000.75")},
        credits={"loans_receivable": Decimal("250000.75")},
        description="Borrower repays principal into cash",
    )
)
assert outcome_repay.pre_rule_rejected is False
fin_next = FinancialState.from_world_state(outcome_repay.next_state)
assert fin_next.accounting_discrepancy == ZERO_DECIMAL
assert fin_next.get_balance("cash") == Decimal("10250000.75")

# 3. Verify that a $0.01 imbalance is rejected by the Pre-Rules gate and rolled back
outcome_bad = fin_engine.step(
    action=build_journal_action(
        "unbalanced_by_plus_0_01",
        debits={"cash": Decimal("1000.00")},
        credits={"retained_earnings": Decimal("1000.01")},
    )
)
assert outcome_bad.pre_rule_rejected is True
assert outcome_bad.rolled_back is True

# 4. Run the 5-step corporate liquidity drain stress test via engine.branch().do()
stress = run_liquidity_drain_stress_test(steps=5)
assert stress.all_steps_balanced is True
assert stress.max_abs_discrepancy_across_all_branches == ZERO_DECIMAL
print(
    f"Section 2 verified: Baseline cash=${stress.baseline_final_cash}, "
    f"Stress cash=${stress.stress_final_cash}, Recovery cash=${stress.recovery_final_cash}, "
    f"max discrepancy=${stress.max_abs_discrepancy_across_all_branches}."
)

## 3. Software Engineering (SWE) Domain Pack (`EWM-005`): Dependency DAG & Concurrent Agent Intervention

In [3]:
from ewm_engine.domains.swe import (
    CircularDependencyError,
    CodeFileNode,
    PullRequestStatus,
    StructuralViolationError,
    SWEState,
    build_swe_action,
    build_swe_agent_engine,
    create_default_swe_repository_state,
    run_concurrent_agent_intervention_scenario,
)

# 1. Verify that constructing a cyclic dependency graph raises CircularDependencyError
cycle_rejected = False
try:
    SWEState(
        files={
            "a.py": CodeFileNode(file_path="a.py"),
            "b.py": CodeFileNode(file_path="b.py"),
        },
        dependencies={"a.py": ("b.py",), "b.py": ("a.py",)},
    )
except CircularDependencyError as exc:
    cycle_rejected = exc.cycle_path == ("a.py", "b.py", "a.py")
assert cycle_rejected is True

# 2. Verify that adding a cyclic dependency edge via an agent action raises StructuralViolationError
swe_engine = build_swe_agent_engine()
base_world = create_default_swe_repository_state().to_world_state()
structural_rejected = False
try:
    swe_engine.step(
        base_world,
        build_swe_action(
            "add_cycle",
            "ADD_DEPENDENCY",
            actor_id="agent_x",
            parameters={
                "source_file": "src/core/schema.py",
                "target_file": "src/api/OrderController.py",
            },
        ),
    )
except StructuralViolationError:
    structural_rejected = True
assert structural_rejected is True

# 3. Execute the concurrent multi-agent PR merge vs. file modification intervention scenario
swe_trace = run_concurrent_agent_intervention_scenario()
assert swe_trace.branch_agent_a_merges_first.pr_statuses["PR-101"] == PullRequestStatus.MERGED.value
assert swe_trace.branch_agent_b_modifies_first.pr_statuses["PR-101"] == PullRequestStatus.CONFLICT_STALE.value
assert swe_trace.cyclic_injection_rejected is True
print(
    f"Section 3 verified: Branch A PR-101={swe_trace.branch_agent_a_merges_first.pr_statuses['PR-101']}, "
    f"Branch B PR-101={swe_trace.branch_agent_b_modifies_first.pr_statuses['PR-101']}."
)

## 4. Scientific Evaluation Framework (`EWM-006`): `EWM_Evaluator` Across $N = 30$ Seeds

In [4]:
from ewm_engine.evaluation.evaluator import (
    MIN_SIGNIFICANCE_SEEDS,
    EWM_Evaluator,
    EnvironmentSpec,
)

evaluator = EWM_Evaluator(
    env_spec=EnvironmentSpec(horizon_steps=10),
    timeout_seconds=30.0,
)
eval_report = evaluator.evaluate(n_seeds=MIN_SIGNIFICANCE_SEEDS, master_seed=42)

assert eval_report.significance_claim_permitted is True
assert eval_report.agent_summaries["EWMPolicyAgent"].constraint_violation_rate.mean == 0.0
assert (
    eval_report.agent_summaries["EWMPolicyAgent"].cumulative_reward.iqm
    > eval_report.agent_summaries["StaticRuleAgent"].cumulative_reward.iqm
)

for agent_id, summary in eval_report.agent_summaries.items():
    print(
        f"{agent_id:16s} | IQM Reward: {summary.cumulative_reward.iqm:8.2f} "
        f"[{summary.cumulative_reward.ci_lower_95:8.2f}, {summary.cumulative_reward.ci_upper_95:8.2f}] | "
        f"Violation Rate: {summary.constraint_violation_rate.mean:.4f}"
    )